In [1]:
import roboticstoolbox as rtb
import swift
import numpy as np
import cv2
from scipy.spatial.transform import Rotation as R
from spatialgeometry import Sphere


ModuleNotFoundError: No module named 'cv2'

In [ ]:

# Crea il robot Frankie
robot = rtb.models.URDF.Frankie()

# Inizia il simulatore Swift
env = swift.Swift()
env.launch(realtime=True,comms="rtc",browser="notebook")

# Aggiungi il robot
env.add(robot)

# Definisci la posizione della camera rispetto all'end effector
# La camera guarda lungo l'asse z dell'end effector
camera_offset = [0, 0, 0.1]  # 10 cm sopra l'end effector


In [ ]:
def get_camera_view(robot, env, camera_offset):
    # Ottieni la posa corrente dell'end effector
    Te = robot.fkine(robot.q)
    
    # Posizione della camera (traslata rispetto all'end effector)
    camera_pos = Te.t + camera_offset
    
    # Orientamento della camera (allineato con l'end effector)
    camera_orient = Te.R
    
    # Ottieni l'immagine dalla camera
    image = env.get_image(
        camera_pos=camera_pos,
        camera_target=Te.t,  # Punta verso l'end effector
        camera_orientation=camera_orient,
        width=640,
        height=480,
        fov=60
    )
    
    return image

In [ ]:
# Aggiungi una sfera colorata nella scena

# Crea una sfera rossa
sphere = Sphere(radius=0.05, color="red")
sphere.T = np.array([
    [1, 0, 0, 0.5],   # Posizione x=0.5m
    [0, 1, 0, 0.2],   # Posizione y=0.2m  
    [0, 0, 1, 0.1],   # Posizione z=0.1m
    [0, 0, 0, 1]
])

env.add(sphere)

In [ ]:
def detect_sphere_color(image):
    # Converti l'immagine in formato OpenCV
    img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    # Converti in HSV per una migliore segmentazione del colore
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    
    # Definisci i range di colore per rosso (HSV)
    lower_red1 = np.array([0, 120, 70])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 120, 70])
    upper_red2 = np.array([180, 255, 255])
    
    # Crea maschere per il rosso
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    red_mask = mask1 + mask2
    
    # Trova i contorni
    contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    sphere_detected = False
    sphere_position = None
    
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > 100:  # Filtra per area minima
            sphere_detected = True
            # Calcola il centro del contorno
            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                sphere_position = (cx, cy)
            
            # Disegna il contorno
            cv2.drawContours(img_bgr, [contour], -1, (0, 255, 0), 2)
    
    return img_bgr, sphere_detected, sphere_position, red_mask

In [ ]:
# Movimento del robot per testare la visione
import time

# Definisci alcune pose di test
q1 = [0, 0, 0, 0, 0, 0]  # Posizione home
q2 = [0.5, 0.3, 0.2, 0, 0, 0]  # Punto di vista migliore

# Esegui il movimento e monitora la sfera
for q_target in [q1, q2]:
    # Muovi il robot
    traj = rtb.jtraj(robot.q, q_target, 50)
    
    for q in traj.q:
        robot.q = q
        env.step(0.05)
        
        # Ottieni la vista della camera
        image = get_camera_view(robot, env, camera_offset)
        
        # Elabora l'immagine per rilevare la sfera
        processed_img, detected, position, mask = detect_sphere_color(image)
        
        if detected:
            print(f"Sfera rilevata alla posizione: {position}")
            # Aggiungi testo all'immagine
            cv2.putText(processed_img, "SPHERE DETECTED", (50, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Mostra le immagini
        cv2.imshow('Camera View', processed_img)
        cv2.imshow('Color Mask', mask)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        time.sleep(0.05)

cv2.destroyAllWindows()